# Snowflake Table Scan Efficiency Report

A free, **fully source-available** health check for how efficiently your Snowflake **table scans**
run — how much data they read vs. skip, and what the avoidable scanning is worth. Read every cell —
it does exactly what it says and nothing more.

> **Runs entirely in your account.** This notebook makes **no external network calls**, sends
> **no data anywhere**, and creates **no persistent objects**. It only *reads* Snowflake's own
> telemetry and shows you the results.

> **One term up front:** every Snowflake query reads a table by **scanning its micro-partitions**
> (the `TableScan` step in a query profile), skipping — "pruning" — the ones it doesn't need. So
> *scanned* partitions are data you paid to read; *skipped* (pruned) partitions are data you avoided.
> This report measures how much your table scans still read — i.e. the avoidable work.

### What it tells you
1. **How much of your data Snowflake scans** — the share of micro-partitions read vs. skipped on
   your large tables. Heavy scanning = opportunity.
2. **The dollar size of the opportunity** — a conservative estimate of compute spent scanning data a
   better layout could avoid.
3. **Which tables scan the most unnecessary data**, ranked, with an apportioned dollar figure.
4. **Which columns your queries filter and join on** — per-column, for those tables.

### What it deliberately does *not* do
It does **not** tell you which clustering key to build, in what column order, or project your
post-change savings. That requires modeling selectivity, cardinality, and how predicates interact —
that's what [Goldilox Insights](https://goldilox.com) does. This report sizes the prize; the
engine tells you exactly how to claim it.

### Data sources (all native Snowflake, all read-only)
- `SNOWFLAKE.ACCOUNT_USAGE.TABLE_PRUNING_HISTORY` — per-table micro-partitions scanned vs. skipped.
- `SNOWFLAKE.ACCOUNT_USAGE.COLUMN_QUERY_PRUNING_HISTORY` — per-column filter/join usage.
- `SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY` — credits attributed per query (for the $ estimate).

### Before you run
- **Run as `ACCOUNTADMIN`** (or a role granted `IMPORTED PRIVILEGES` on the `SNOWFLAKE` database).
- **Attach any warehouse** — an `X-SMALL` is plenty; the queries are light.
- **Latency:** these views lag real time by a few hours. They require Enterprise Edition or above;
  if your account doesn't have them, the affected sections will say so.

Set your parameters in the next cell, then **Run All**.

---
## Configuration

Adjust these and run. Set `CREDIT_PRICE` to your Snowflake contract rate for an accurate dollar
estimate. Everything else is safe at the defaults.

In [ ]:
# === Parameters ============================================================
LOOKBACK_DAYS  = 30     # analysis window (days)
MIN_PARTITIONS = 100    # ignore small tables: require avg partitions-per-scan >= this.
                        # Tiny tables have little or nothing to prune.
TOP_N_TABLES   = 20     # tables shown in Section 3 ranking + charts. Bounds page size — the rest
                        # are still counted in the account totals/trend. Keep modest (<= ~50).
COLUMN_DETAIL_TABLES = 8  # of those, how many to break down by column in Section 4
CREDIT_PRICE   = 3.00   # $ per Snowflake credit — set to YOUR contract rate
# ===========================================================================

import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

def q(sql, params=None):
    """Run SQL in this account and return a pandas DataFrame."""
    return (session.sql(sql, params=params) if params else session.sql(sql)).to_pandas()

# Shared across cells (set later). None until computed.
SCAN_CREDITS_ANNUAL = None   # annualized credits attributable to partition scanning
TOTAL_SCANNED       = None   # total partitions scanned across the gated table set (Section 3)

st.success(f"Configuration loaded — analyzing the last {LOOKBACK_DAYS} days "
           f"at ${CREDIT_PRICE:.2f}/credit.")

---
## 1. How much of your data Snowflake scans

The headline. Across your large tables, what share of micro-partitions does Snowflake **scan (read)**
vs. **skip**? A high scan share means filters aren't cutting how much gets read — i.e. real
opportunity. Small tables are excluded (see the partition gate in config).

In [ ]:
st.markdown("## 1. How much of your data Snowflake scans")

try:
    df = q(f"""
        WITH per_table AS (
            SELECT database_name, schema_name, table_name,
                   SUM(num_scans)         AS num_scans,
                   SUM(partitions_scanned) AS ps,
                   SUM(partitions_pruned)  AS pp
            FROM SNOWFLAKE.ACCOUNT_USAGE.TABLE_PRUNING_HISTORY
            WHERE start_time >= DATEADD('day', -{LOOKBACK_DAYS}, CURRENT_TIMESTAMP())
            GROUP BY 1, 2, 3
            -- partition gate: avg partitions considered per scan >= MIN_PARTITIONS
            HAVING (SUM(partitions_scanned) + SUM(partitions_pruned))
                   / NULLIF(SUM(num_scans), 0) >= {MIN_PARTITIONS}
        )
        SELECT
            COUNT(*)            AS table_count,
            SUM(num_scans)      AS num_scans,
            SUM(ps)             AS partitions_scanned,
            SUM(pp)             AS partitions_pruned,
            ROUND(100 * SUM(pp) / NULLIF(SUM(ps) + SUM(pp), 0), 1) AS prune_pct,
            ROUND(100 * SUM(ps) / NULLIF(SUM(ps) + SUM(pp), 0), 1) AS scan_pct
        FROM per_table
    """)
    row = df.iloc[0]
    prune_pct = float(row["PRUNE_PCT"] or 0)
    scan_pct  = float(row["SCAN_PCT"] or 0)
    TOTAL_SCANNED = float(row["PARTITIONS_SCANNED"] or 0)

    c1, c2, c3 = st.columns(3)
    c1.metric("Large tables analyzed", f"{int(row['TABLE_COUNT'] or 0):,}",
              help=f"Tables whose scans consider >= {MIN_PARTITIONS} micro-partitions on average "
                   "((scanned+skipped)/num_scans). A proxy for table size — the scan-history views don't "
                   "expose a partition count. Smaller tables are excluded: little to skip.")
    c2.metric("Partitions scanned (read)", f"{scan_pct:.1f}%",
              help="Share of candidate micro-partitions Snowflake actually read. Lower = better — "
                   "this is the avoidable work.")
    c3.metric("Partitions skipped (pruned)", f"{prune_pct:.1f}%",
              help="Share Snowflake skipped without reading ('pruned'). Higher = better.")
    st.caption(f"\u201cLarge\u201d = a table whose scans consider \u2265 {MIN_PARTITIONS} "
               f"micro-partitions on average (set by MIN_PARTITIONS in config). "
               f"{int(row['TABLE_COUNT'] or 0):,} of your tables qualify; smaller tables are excluded "
               "because they have little to skip.")

    if scan_pct >= 50:
        st.warning(f"Your large tables **read {scan_pct:.0f}%** of the partitions they touch — "
                   "filters are doing little to cut scanning. Likely significant opportunity.")
    elif scan_pct >= 20:
        st.info(f"Your large tables read **{scan_pct:.0f}%** of partitions — moderate opportunity.")
    else:
        st.success(f"Your large tables read only **{scan_pct:.0f}%** of partitions — scanning is "
                   "already fairly efficient.")
except Exception as e:
    st.warning("Could not read `TABLE_PRUNING_HISTORY` (it requires Enterprise Edition or above, "
               "and may not be populated yet). The dollar estimate below still works.\n\n"
               f"`{e}`")


# --- Scanning over time (daily, same large-table set) ---
try:
    import altair as alt
    ts = q(f"""
        WITH large AS (
            SELECT database_name || '.' || schema_name || '.' || table_name AS tbl
            FROM SNOWFLAKE.ACCOUNT_USAGE.TABLE_PRUNING_HISTORY
            WHERE start_time >= DATEADD('day', -{LOOKBACK_DAYS}, CURRENT_TIMESTAMP())
            GROUP BY 1
            HAVING (SUM(partitions_scanned) + SUM(partitions_pruned))
                   / NULLIF(SUM(num_scans), 0) >= {MIN_PARTITIONS}
        )
        SELECT DATE_TRUNC('day', start_time) AS day,
               SUM(partitions_scanned) AS scanned,
               SUM(partitions_pruned)  AS pruned,
               ROUND(100 * SUM(partitions_pruned)
                     / NULLIF(SUM(partitions_scanned) + SUM(partitions_pruned), 0), 1) AS prune_pct
        FROM SNOWFLAKE.ACCOUNT_USAGE.TABLE_PRUNING_HISTORY
        WHERE start_time >= DATEADD('day', -{LOOKBACK_DAYS}, CURRENT_TIMESTAMP())
          AND database_name || '.' || schema_name || '.' || table_name IN (SELECT tbl FROM large)
        GROUP BY 1
        ORDER BY 1
    """)
    if len(ts) > 1:
        st.markdown("**Scanning over time**")
        ts_long = ts.melt(id_vars=["DAY", "PRUNE_PCT"],
                          value_vars=["SCANNED", "PRUNED"],
                          var_name="kind", value_name="partitions")
        ts_long["kind"] = ts_long["kind"].map({"SCANNED": "Scanned (read)",
                                               "PRUNED": "Skipped (pruned)"})
        area = (
            alt.Chart(ts_long).mark_area().encode(
                x=alt.X("DAY:T", title=None),
                y=alt.Y("partitions:Q", title="Micro-partitions / day", stack=True),
                color=alt.Color("kind:N", title=None,
                    scale=alt.Scale(domain=["Scanned (read)", "Skipped (pruned)"],
                                    range=["#d62728", "#2ca02c"]),
                    legend=alt.Legend(orient="top")),
                order=alt.Order("kind:N", sort="descending"),
                tooltip=[alt.Tooltip("DAY:T", title="day"),
                         alt.Tooltip("kind:N", title=""),
                         alt.Tooltip("partitions:Q", title="partitions", format=","),
                         alt.Tooltip("PRUNE_PCT:Q", title="prune % (day)")])
            .properties(height=240)
        )
        st.altair_chart(area, use_container_width=True)
        st.caption("Daily micro-partitions read (red) vs skipped (green) across your large tables. "
                   "A persistent or growing red band = avoidable scanning that isn't improving on its own.")
except Exception:
    pass


---
## 2. Scan-attributable compute (the opportunity, in dollars)

A **conservative upper bound** on the compute tied up in scanning data, annualized from your sample
window. This is the size of the prize, not a guaranteed saving. See **Methodology** for the formula.

In [ ]:
st.markdown("## 2. Scan-attributable compute (opportunity estimate)")

try:
    df2 = q(f"""
        WITH base AS (
            SELECT query_id, partitions_scanned, partitions_total
            FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
            WHERE query_type = 'SELECT'
              AND execution_status = 'SUCCESS'
              AND partitions_total >= {MIN_PARTITIONS}
              AND partitions_scanned IS NOT NULL
              AND start_time >= DATEADD('day', -{LOOKBACK_DAYS}, CURRENT_TIMESTAMP())
        ),
        attr AS (
            SELECT query_id, SUM(credits_attributed_compute) AS credits
            FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY
            WHERE start_time >= DATEADD('day', -{LOOKBACK_DAYS}, CURRENT_TIMESTAMP())
            GROUP BY query_id
        )
        SELECT
            SUM(a.credits)                                                        AS total_credits,
            SUM(a.credits * b.partitions_scanned / NULLIF(b.partitions_total, 0)) AS scan_credits
        FROM base b
        JOIN attr a USING (query_id)
    """)
    total_credits = float(df2.iloc[0]["TOTAL_CREDITS"] or 0)
    scan_credits  = float(df2.iloc[0]["SCAN_CREDITS"] or 0)
    SCAN_CREDITS_ANNUAL = scan_credits * 365.0 / LOOKBACK_DAYS

    c1, c2 = st.columns(2)
    c1.metric("Scan-attributable credits (annualized)", f"{SCAN_CREDITS_ANNUAL:,.0f}")
    c2.metric("Est. annualized opportunity", f"${SCAN_CREDITS_ANNUAL * CREDIT_PRICE:,.0f}",
              help="Credits attributed to filtered scans, weighted by the fraction of partitions "
                   "read, annualized. Conservative UPPER BOUND — not a guaranteed saving.")
    pct = (100 * scan_credits / total_credits) if total_credits else 0
    st.caption(f"Over the {LOOKBACK_DAYS}-day window, {scan_credits:,.0f} credits were attributable "
               f"to partition scanning on filtered queries ({pct:.0f}% of those queries' compute), "
               f"at ${CREDIT_PRICE:.2f}/credit. See Methodology for assumptions.")
except Exception as e:
    st.warning("Could not read `QUERY_ATTRIBUTION_HISTORY` (it may not be populated yet). "
               "Skipping the dollar estimate — the pruning sections still apply.\n\n"
               f"`{e}`")

---
## 3. Tables scanning the most unnecessary data

Per-table opportunity, straight from Snowflake's table scan telemetry — no joins, no sampling.
Ranked by scan volume × scan fraction. If the dollar estimate above succeeded, each table's share of
the opportunity is apportioned by its share of scanned partitions.

In [ ]:
st.markdown("## 3. Tables scanning the most unnecessary data")
top_tables = []

try:
    tdf = q(f"""
        WITH agg AS (
            SELECT
                database_name || '.' || schema_name || '.' || table_name AS table_name,
                SUM(num_scans)          AS num_scans,
                SUM(partitions_scanned) AS ps,
                SUM(partitions_pruned)  AS pp,
                SUM(rows_scanned)       AS rows_scanned
            FROM SNOWFLAKE.ACCOUNT_USAGE.TABLE_PRUNING_HISTORY
            WHERE start_time >= DATEADD('day', -{LOOKBACK_DAYS}, CURRENT_TIMESTAMP())
            GROUP BY 1
            -- partition gate: avg partitions considered per scan >= MIN_PARTITIONS
            HAVING (SUM(partitions_scanned) + SUM(partitions_pruned))
                   / NULLIF(SUM(num_scans), 0) >= {MIN_PARTITIONS}
        )
        SELECT
            table_name,
            num_scans,
            ps                                       AS partitions_scanned,
            pp                                       AS partitions_pruned,
            ROUND(100 * pp / NULLIF(ps + pp, 0), 1)  AS prune_pct,
            ROUND(100 * ps / NULLIF(ps + pp, 0), 1)  AS scan_pct,
            ROUND(rows_scanned / 1e9, 2)             AS billion_rows_scanned
        FROM agg
        ORDER BY ps * ps / NULLIF(ps + pp, 0) DESC   -- scan volume x scan fraction
        LIMIT {TOP_N_TABLES}
    """)

    if len(tdf):
        # Apportion the annualized $ opportunity by share of scanned partitions.
        if SCAN_CREDITS_ANNUAL and TOTAL_SCANNED:
            opp_total = SCAN_CREDITS_ANNUAL * CREDIT_PRICE
            tdf["EST_ANNUAL_OPP_USD"] = (
                (tdf["PARTITIONS_SCANNED"] / TOTAL_SCANNED) * opp_total
            ).round(0)

        st.dataframe(tdf, use_container_width=True, hide_index=True)
        st.caption("Ranked by scan volume × scan fraction — the biggest opportunities are at the "
                   "top. The dollar column (if present) apportions Section 2's estimate by each "
                   "table's share of scanned partitions.")

        # Headroom chart: per table, candidate micro-partitions split into read (opportunity)
        # vs skipped (working pruning), ordered by opportunity, biggest at top.
        try:
            import altair as alt
            cdf_chart = tdf.copy()
            cdf_chart["LABEL"] = cdf_chart["TABLE_NAME"].apply(lambda t: ".".join(t.split(".")[-2:]))
            order = cdf_chart["LABEL"].tolist()
            idv = ["LABEL", "TABLE_NAME", "SCAN_PCT", "PRUNE_PCT", "NUM_SCANS"]
            tips = [alt.Tooltip("TABLE_NAME:N", title="table"), alt.Tooltip("kind:N", title=""),
                    alt.Tooltip("partitions:Q", title="partitions", format=","),
                    alt.Tooltip("SCAN_PCT:Q", title="scan %"),
                    alt.Tooltip("PRUNE_PCT:Q", title="prune %"),
                    alt.Tooltip("NUM_SCANS:Q", title="# scans", format=",")]
            if "EST_ANNUAL_OPP_USD" in cdf_chart.columns:
                idv.append("EST_ANNUAL_OPP_USD")
                tips.append(alt.Tooltip("EST_ANNUAL_OPP_USD:Q", title="est $/yr", format="$,.0f"))
            long_df = cdf_chart.melt(
                id_vars=idv, value_vars=["PARTITIONS_SCANNED", "PARTITIONS_PRUNED"],
                var_name="kind", value_name="partitions")
            long_df["kind"] = long_df["kind"].map({
                "PARTITIONS_SCANNED": "Scanned (read)",
                "PARTITIONS_PRUNED": "Skipped (pruned)"})
            chart = (
                alt.Chart(long_df).mark_bar().encode(
                    y=alt.Y("LABEL:N", sort=order, title=None),
                    x=alt.X("partitions:Q", title="Candidate micro-partitions: read vs skipped",
                            stack=True),
                    color=alt.Color("kind:N", title=None,
                        scale=alt.Scale(domain=["Scanned (read)", "Skipped (pruned)"],
                                        range=["#d62728", "#2ca02c"]),
                        legend=alt.Legend(orient="top")),
                    order=alt.Order("kind:N", sort="descending"),
                    tooltip=tips)
                .properties(height=max(220, 24 * len(order)))
            )
            st.altair_chart(chart, use_container_width=True)
            st.caption("Each bar is a table's candidate micro-partitions, split into **read** (red — "
                       "the opportunity) and **skipped** (green — scanning Snowflake already avoids). "
                       "A long red segment = lots of avoidable scanning. Hover for scan %, skip %, est $/yr.")
        except Exception as _chart_err:
            st.bar_chart(tdf.set_index("TABLE_NAME")["PARTITIONS_SCANNED"])
            st.caption(f"(simple chart fallback: {_chart_err})")

        # Dollar-opportunity bar — only when Section 2 produced an estimate.
        if "EST_ANNUAL_OPP_USD" in tdf.columns and tdf["EST_ANNUAL_OPP_USD"].sum() > 0:
            try:
                import altair as alt
                ddf = tdf.copy()
                ddf["LABEL"] = ddf["TABLE_NAME"].apply(lambda t: ".".join(t.split(".")[-2:]))
                dorder = ddf.sort_values("EST_ANNUAL_OPP_USD", ascending=False)["LABEL"].tolist()
                dchart = (
                    alt.Chart(ddf).mark_bar(color="#1f77b4").encode(
                        y=alt.Y("LABEL:N", sort=dorder, title=None),
                        x=alt.X("EST_ANNUAL_OPP_USD:Q",
                                title="Est. annualized opportunity ($)",
                                axis=alt.Axis(format="$,.0f")),
                        tooltip=[alt.Tooltip("TABLE_NAME:N", title="table"),
                                 alt.Tooltip("EST_ANNUAL_OPP_USD:Q", title="est $/yr", format="$,.0f"),
                                 alt.Tooltip("SCAN_PCT:Q", title="scan %"),
                                 alt.Tooltip("PRUNE_PCT:Q", title="prune %")])
                    .properties(height=max(220, 24 * len(ddf)))
                )
                st.markdown("**Estimated annualized opportunity by table**")
                st.altair_chart(dchart, use_container_width=True)
                st.caption("Section 2's annualized $ estimate apportioned to each table by its share "
                           "of scanned partitions. Upper-bound sizing — see Methodology.")
            except Exception:
                pass

        top_tables = tdf["TABLE_NAME"].tolist()
    else:
        st.info("No tables passed the partition gate in this window. Try a larger LOOKBACK_DAYS or "
                "a smaller MIN_PARTITIONS.")
except Exception as e:
    st.warning("Could not read `TABLE_PRUNING_HISTORY`. Per-table ranking skipped — Section 2 "
               f"still applies.\n\n`{e}`")

---
## 4. Columns your queries filter and join on

Straight from `COLUMN_QUERY_PRUNING_HISTORY`: for your top tables, which columns appear in `WHERE`
and `JOIN` predicates, and how often. This is **observed filter/join frequency** — where a clustering
strategy would focus — **not a clustering-key recommendation**.

> **Why no per-column prune rate?** Snowflake attributes each query's *table-level* pruning to **every**
> predicate column in that query, so a "per-column prune %" reflects the whole query, not the column's
> own effect (an unrelated measure column shows the same pruned-partition count as the join key it ran
> alongside). Real pruning is a table property — see Section 3.

In [ ]:
st.markdown("## 4. Columns your queries filter and join on")
st.caption("Which columns appear in WHERE / JOIN predicates on your top tables, and how often — "
           "**not** a clustering-key recommendation, and not a per-column prune rate (see note above).")

if not top_tables:
    st.info("No candidate tables from Section 3 — nothing to inspect.")
else:
    detail_tables = top_tables[:COLUMN_DETAIL_TABLES]
    in_list = ", ".join("'%s'" % t.replace("'", "''") for t in detail_tables)
    try:
        cdf = q(f"""
            SELECT
                database_name || '.' || schema_name || '.' || table_name AS table_name,
                column_name,
                access_type,
                SUM(num_queries) AS num_queries
            FROM SNOWFLAKE.ACCOUNT_USAGE.COLUMN_QUERY_PRUNING_HISTORY
            WHERE interval_start_time >= DATEADD('day', -{LOOKBACK_DAYS}, CURRENT_TIMESTAMP())
              AND database_name || '.' || schema_name || '.' || table_name IN ({in_list})
            GROUP BY 1, 2, 3
            HAVING SUM(num_queries) > 0
            ORDER BY table_name, num_queries DESC
        """)
        if len(cdf):
            st.caption(f"Showing the top {len(detail_tables)} of {len(top_tables)} ranked tables.")
            for tbl in detail_tables:
                sub = cdf[cdf["TABLE_NAME"] == tbl]
                if not len(sub):
                    continue
                st.markdown(f"**{tbl}**")
                st.dataframe(sub[["COLUMN_NAME", "ACCESS_TYPE", "NUM_QUERIES"]].head(12),
                             use_container_width=True, hide_index=True)
            st.caption("`NUM_QUERIES` = how many queries referenced that column in a WHERE/JOIN "
                       "predicate. The most-referenced columns are the natural focus for pruning work; "
                       "table-level pruning headroom is in Section 3.")
        else:
            # Top-scan tables had no filter columns (often full-table scans). Fall back to the
            # most-referenced filter/join columns account-wide so the section is still informative.
            fb = q(f"""
                SELECT database_name || '.' || schema_name || '.' || table_name AS table_name,
                       column_name, access_type, SUM(num_queries) AS num_queries
                FROM SNOWFLAKE.ACCOUNT_USAGE.COLUMN_QUERY_PRUNING_HISTORY
                WHERE interval_start_time >= DATEADD('day', -{LOOKBACK_DAYS}, CURRENT_TIMESTAMP())
                GROUP BY 1, 2, 3
                ORDER BY num_queries DESC
                LIMIT 20
            """)
            if len(fb):
                st.caption("Your highest-scan tables had no column-level filter history in this window "
                           "— often a sign of full-table scans. Showing your most-referenced filter/join "
                           "columns account-wide instead:")
                st.dataframe(fb, use_container_width=True, hide_index=True)
            else:
                st.info("No column-level pruning history found in the window.")
    except Exception as e:
        st.warning("Could not read `COLUMN_QUERY_PRUNING_HISTORY` (Enterprise Edition or above "
                   f"required). Column-level hints skipped.\n\n`{e}`")

---
## Summary & next step

You now have, for your own account and computed entirely in it:

- the **share of data your large tables scan** vs. skip (Section 1),
- a **conservative dollar opportunity** from cutting avoidable scanning (Section 2),
- the **tables** where that opportunity concentrates, with apportioned dollars (Section 3), and
- the **columns** those queries filter and join on (Section 4).

**If the number is meaningful, the next question is *which clustering key to build, in what order,
and what you'll actually save.*** That's the part this report intentionally doesn't answer — it
requires modeling selectivity, cardinality and predicate interactions per table.

[**Goldilox Insights**](https://goldilox.com) does exactly that: it recommends the specific key
(or Search Optimization / materialized view), projects the post-change savings and ROI, and runs as
a Snowflake Native App — your data never leaves your account. Run this report, see your number,
then let the engine tell you how to claim it.

*Questions about the methodology below? Email support@goldilox.com.*

---
## Methodology & caveats

**Data sources** — all native Snowflake, all read-only, all in your account:
- `SNOWFLAKE.ACCOUNT_USAGE.TABLE_PRUNING_HISTORY` — per-table `PARTITIONS_SCANNED` / `PARTITIONS_PRUNED`
  / `ROWS_*` / `NUM_SCANS`, hourly. Latency ~6h. (Enterprise Edition+.)
- `SNOWFLAKE.ACCOUNT_USAGE.COLUMN_QUERY_PRUNING_HISTORY` — the same metrics per **column** and
  `ACCESS_TYPE` (WHERE/JOIN), hourly, 1-year retention. Latency ~4h. (Enterprise Edition+.)
- `SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY` + `QUERY_ATTRIBUTION_HISTORY` — per-query partitions and
  attributed credits, for the dollar estimate.

**Partition gate.** A table is included only if its average partitions-considered-per-scan
(`(scanned + pruned) / num_scans`) is at least `MIN_PARTITIONS`. This excludes small tables, which
have little or nothing to prune and would otherwise distort the result.

**Section 1 — scan % / skip %.** `scan% = scanned / (scanned + skipped)`, `skip% = skipped /
(scanned + skipped)`, summed across the gated tables. This describes *current* behavior; it is **not**
a claim about an achievable target (that depends on the right key, which this report doesn't choose). The **over-time** chart applies the same metric per day across the same gated tables.

**Section 2 — opportunity ($).** For each filtered `SELECT`, `credits × (partitions_scanned /
partitions_total)` = the compute tied to the partitions it scanned. Summed, annualized
(`× 365 / LOOKBACK_DAYS`), then `× CREDIT_PRICE`. A deliberately **conservative upper bound** on what
perfect pruning could avoid — real savings depend on how prunable each predicate is. Set
`CREDIT_PRICE` to your contract rate.

**Section 3 — per-table.** Aggregated directly from `TABLE_PRUNING_HISTORY`; ranked by scan volume ×
scan fraction. The dollar column apportions Section 2's annualized estimate across tables by each
table's share of total scanned partitions — an approximation, shown for relative sizing.

**Section 4 — column frequency.** Aggregated from `COLUMN_QUERY_PRUNING_HISTORY` for the top tables.
`NUM_QUERIES` is how many queries referenced each column in a WHERE/JOIN predicate. We deliberately do
**not** show a per-column prune rate: this view attributes each query's *table-level* pruning to every
predicate column in the query (verified — unrelated measure columns report the same pruned-partition
counts as the join keys), so a per-column prune % reflects the whole query, not the column. Table-level
pruning is in Section 3. Observational context, **not** a recommended key.

**Caveats.**
- `ACCOUNT_USAGE` lags real time by a few hours.
- The pruning views require Enterprise Edition or above; without them, those sections are skipped
  and the dollar estimate (Section 2) still works on any edition.
- Hybrid tables are excluded from `TABLE_PRUNING_HISTORY`.
- Numbers reflect the sampled window; widen `LOOKBACK_DAYS` for a steadier estimate.

**No data leaves your account.** This notebook issues only the read queries above and renders the
results inline. It makes no external network calls and creates no persistent objects.